In [1]:
# Import library yang diperlukan
import cv2
import mediapipe as mp
import numpy as np
import threading
import time

In [2]:
def overlay_transparent(bg, overlay, x, y, overlay_size=None):
    """
    Overlay a transparent image on top of a background image.

    Args:
        bg (numpy.ndarray): Background image.
        overlay (numpy.ndarray): Overlay image with alpha channel.
        x (int): X-coordinate of the overlay position.
        y (int): Y-coordinate of the overlay position.
        overlay_size (tuple): Optional size to resize the overlay.

    Returns:
        numpy.ndarray: Combined image with overlay applied.
    """
    bg = bg.copy()
    if overlay_size:
        overlay = cv2.resize(overlay, overlay_size, interpolation=cv2.INTER_AREA)
    h, w = overlay.shape[:2]
    if x + w > bg.shape[1] or y + h > bg.shape[0] or x < 0 or y < 0:
        return bg
    overlay_img = overlay[:, :, :3]
    mask = overlay[:, :, 3] / 255.0
    mask = np.stack([mask] * 3, axis=-1)
    roi = bg[y:y+h, x:x+w]
    blended = (1.0 - mask) * roi + mask * overlay_img
    bg[y:y+h, x:x+w] = blended.astype(np.uint8)
    return bg

In [ ]:
def eye_aspect_ratio(landmarks, eye_indices, img_w, img_h):
    """
    Calculate the Eye Aspect Ratio (EAR) based on landmarks.

    Args:
        landmarks (list): List of facial landmarks.
        eye_indices (list): Indices of eye landmarks.
        img_w (int): Image width.
        img_h (int): Image height.

    Returns:
        float: Calculated EAR value.
    """
    p = [landmarks[i] for i in eye_indices]
    p = [(int(pt.x * img_w), int(pt.y * img_h)) for pt in p]
    A = np.linalg.norm(np.array(p[1]) - np.array(p[5]))
    B = np.linalg.norm(np.array(p[2]) - np.array(p[4]))
    C = np.linalg.norm(np.array(p[0]) - np.array(p[3]))
    ear = (A + B) / (2.0 * C)
    return ear

def draw_face_bounding_box(frame, face_landmarks, img_width, img_height):
    """
    Draw a bounding box around the detected face.

    Args:
        frame (numpy.ndarray): Frame to draw on.
        face_landmarks (list): Detected face landmarks.
        img_width (int): Width of the image/frame.
        img_height (int): Height of the image/frame.

    Returns:
        numpy.ndarray: Frame with bounding box drawn.
    """
    x_coords = [int(landmark.x * img_width) for landmark in face_landmarks.landmark]
    y_coords = [int(landmark.y * img_height) for landmark in face_landmarks.landmark]
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)
    return frame

def draw_face_landmarks(frame, face_landmarks, img_width, img_height):
    """
    Draw facial landmarks on the frame.

    Args:
        frame (numpy.ndarray): Frame to draw on.
        face_landmarks (list): Detected face landmarks.
        img_width (int): Width of the image/frame.
        img_height (int): Height of the image/frame.

    Returns:
        numpy.ndarray: Frame with landmarks drawn.
    """
    for landmark in face_landmarks.landmark:
        x = int(landmark.x * img_width)
        y = int(landmark.y * img_height)
        cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)
    return frame

def detect_hand_gesture(hand_landmarks):
    """
    Detect hand gestures based on landmarks.

    Args:
        hand_landmarks (list): List of hand landmarks.

    Returns:
        str: Gesture name (e.g., "thumbs_up", "open_palm").
    """
    thumb_tip = hand_landmarks.landmark[4]
    index_tip = hand_landmarks.landmark[8]
    middle_tip = hand_landmarks.landmark[12]

    # Contoh: Deteksi thumbs up
    if thumb_tip.y < index_tip.y and thumb_tip.y < middle_tip.y:
        return "thumbs_up"

    return None

In [4]:
def eye_aspect_ratio(landmarks, eye_indices, img_w, img_h):
    """
    Calculate the Eye Aspect Ratio (EAR) based on landmarks.

    Args:
        landmarks (list): List of facial landmarks.
        eye_indices (list): Indices of eye landmarks.
        img_w (int): Image width.
        img_h (int): Image height.

    Returns:
        float: Calculated EAR value.
    """
    p = [landmarks[i] for i in eye_indices]
    p = [(int(pt.x * img_w), int(pt.y * img_h)) for pt in p]
    A = np.linalg.norm(np.array(p[1]) - np.array(p[5]))
    B = np.linalg.norm(np.array(p[2]) - np.array(p[4]))
    C = np.linalg.norm(np.array(p[0]) - np.array(p[3]))
    ear = (A + B) / (2.0 * C)
    return ear

In [5]:
def draw_healthbar(frame, player_id, health, x, y, w=100, h=10):
    """
    Draw a healthbar above the player's head.

    Args:
        frame (numpy.ndarray): Frame to draw on.
        player_id (str): Player ID ("Player 1" or "Player 2").
        health (int): Current health value (0-100).
        x (int): X-coordinate of the healthbar.
        y (int): Y-coordinate of the healthbar.
        w (int): Width of the healthbar.
        h (int): Height of the healthbar.

    Returns:
        numpy.ndarray: Frame with healthbar drawn.
    """
    # Draw background of the healthbar
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 0), 2)
    # Draw filled part of the healthbar
    fill_width = int((health / 100) * w)
    cv2.rectangle(frame, (x, y), (x + fill_width, y + h), (0, 255, 0), -1)
    # Add player label
    cv2.putText(frame, player_id, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return frame

In [ ]:
# Inisialisasi Face Mesh dan Hand Detection
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=2, refine_landmarks=True)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2)

# Load sprite player, peluru, dan shield
player_img = cv2.imread('asset player1/player 1.png', cv2.IMREAD_UNCHANGED)
ammo_img = cv2.imread('asset player1/ammo1.png', cv2.IMREAD_UNCHANGED)
shield_img = cv2.imread('asset player1/shield.png', cv2.IMREAD_UNCHANGED)

if player_img is None or ammo_img is None or shield_img is None:
    raise FileNotFoundError("Gambar tidak ditemukan.")
if player_img.shape[2] < 4 or ammo_img.shape[2] < 4 or shield_img.shape[2] < 4:
    raise ValueError("Gambar tidak memiliki alpha channel.")

# Indeks landmark mata
LEFT_EYE_IDX = [362, 385, 387, 263, 373, 380]
RIGHT_EYE_IDX = [33, 160, 158, 133, 153, 144]

# Landmark hidung untuk posisi player
NOSE_IDX = 1

# Threshold dan cooldown
BLINK_THRESHOLD = 0.15
OPEN_THRESHOLD = 0.25
BLINK_COOLDOWN = 1.0

# Ukuran player
PLAYER_W = 100
PLAYER_H = int(PLAYER_W * player_img.shape[0] / player_img.shape[1])

# Status shield
shield_active_player1 = False
shield_active_player2 = False

def activate_shield(player_id):
    """
    Activate shield for the specified player.

    Args:
        player_id (str): "Player 1" or "Player 2".
    """
    global shield_active_player1, shield_active_player2
    if player_id == "Player 1":
        shield_active_player1 = True
        print("Player 1 activated SHIELD!")
    elif player_id == "Player 2":
        shield_active_player2 = True
        print("Player 2 activated SHIELD!")

    # Matikan shield setelah beberapa detik
    def deactivate_shield():
        nonlocal player_id
        time.sleep(5)  # Shield aktif selama 5 detik
        if player_id == "Player 1":
            shield_active_player1 = False
        elif player_id == "Player 2":
            shield_active_player2 = False
        print(f"{player_id} shield deactivated.")

    threading.Thread(target=deactivate_shield).start()

def main():
    """
    Main function to run the interactive blink-based shooting game with two players.
    """
    # Buka kamera
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise IOError("Tidak dapat membuka kamera.")

    # Inisialisasi variabel
    last_blink_time = [0, 0]
    eye_ready_to_blink = [0, 0]
    projectiles = []
    health_player1 = 100
    health_player2 = 100

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        ih, iw = frame.shape[:2]
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results_face = face_mesh.process(rgb_frame)
        results_hands = hands.process(rgb_frame)
        current_time = time.time()

        # Reset proyektil yang keluar dari layar
        new_projectiles = []
        for proj in projectiles:
            proj['x'] += 10 if proj['player'] == "Player 1" else -10
            if 0 <= proj['x'] <= iw:
                new_projectiles.append(proj)
        projectiles = new_projectiles

        if results_face.multi_face_landmarks:
            for idx, face_landmarks in enumerate(results_face.multi_face_landmarks[:2]):
                # Gambar bounding box dan landmark points
                frame = draw_face_bounding_box(frame, face_landmarks, iw, ih)
                frame = draw_face_landmarks(frame, face_landmarks, iw, ih)

                # Deteksi kedipan mata
                left_ear = eye_aspect_ratio(face_landmarks.landmark, LEFT_EYE_IDX, iw, ih)
                right_ear = eye_aspect_ratio(face_landmarks.landmark, RIGHT_EYE_IDX, iw, ih)
                avg_ear = (left_ear + right_ear) / 2.0

                nose = face_landmarks.landmark[NOSE_IDX]
                player_x = int(nose.x * iw) - PLAYER_W // 2
                player_y = int(nose.y * ih) - PLAYER_H // 2

                blinking = avg_ear < BLINK_THRESHOLD
                eyes_open = avg_ear > OPEN_THRESHOLD

                if eyes_open:
                    eye_ready_to_blink[idx] = 1

                if blinking and eye_ready_to_blink[idx] == 1 and current_time - last_blink_time[idx] >= BLINK_COOLDOWN:
                    # Tembakkan peluru
                    ammo_w = 30
                    ammo_h = int(ammo_w * ammo_img.shape[0] / ammo_img.shape[1])
                    ammo_start_x = player_x + PLAYER_W if idx == 0 else player_x - ammo_w
                    ammo_start_y = player_y + PLAYER_H // 2 - ammo_h // 2
                    projectiles.append({
                        'x': ammo_start_x,
                        'y': ammo_start_y,
                        'w': ammo_w,
                        'h': ammo_h,
                        'player': f"Player {idx + 1}"
                    })
                    last_blink_time[idx] = current_time
                    eye_ready_to_blink[idx] = 0

                # Gambar player
                frame = overlay_transparent(frame, player_img, player_x, player_y, (PLAYER_W, PLAYER_H))

                # Gambar healthbar
                health = health_player1 if idx == 0 else health_player2
                frame = draw_healthbar(frame, f"Player {idx + 1}", health, player_x, player_y - 20)

        # Deteksi gestur tangan
        if results_hands.multi_hand_landmarks:
            for hand_landmarks in results_hands.multi_hand_landmarks:
                gesture = detect_hand_gesture(hand_landmarks)
                if gesture == "thumbs_up":
                    # Aktifkan shield untuk pemain yang melakukan gestur
                    if hand_landmarks.landmark[0].x < 0.5:  # Pemain di kiri frame
                        activate_shield("Player 1")
                    else:  # Pemain di kanan frame
                        activate_shield("Player 2")

        # Gambar proyektil dan deteksi tabrakan
        for proj in projectiles:
            frame = overlay_transparent(frame, ammo_img, proj['x'], proj['y'], (proj['w'], proj['h']))

            # Deteksi tabrakan dengan player
            for idx, face_landmarks in enumerate(results_face.multi_face_landmarks[:2]):
                nose = face_landmarks.landmark[NOSE_IDX]
                player_x = int(nose.x * iw) - PLAYER_W // 2
                player_y = int(nose.y * ih) - PLAYER_H // 2

                if (player_x < proj['x'] < player_x + PLAYER_W and
                    player_y < proj['y'] < player_y + PLAYER_H and
                    proj['player'] != f"Player {idx + 1}" and
                    not (shield_active_player1 if idx == 0 else shield_active_player2)):
                    if idx == 0:
                        health_player1 -= 10
                    else:
                        health_player2 -= 10
                    projectiles.remove(proj)

        # Gambar shield jika aktif
        for idx, face_landmarks in enumerate(results_face.multi_face_landmarks[:2]):
            nose = face_landmarks.landmark[NOSE_IDX]
            player_x = int(nose.x * iw) - PLAYER_W // 2
            player_y = int(nose.y * ih) - PLAYER_H // 2

            if idx == 0 and shield_active_player1:
                frame = overlay_transparent(frame, shield_img, player_x, player_y, (PLAYER_W, PLAYER_H))
            elif idx == 1 and shield_active_player2:
                frame = overlay_transparent(frame, shield_img, player_x, player_y, (PLAYER_W, PLAYER_H))

        # Tampilkan frame
        cv2.imshow("Two Player Blink Shot", frame)
        if cv2.waitKey(5) & 0xFF == 27:  # Tekan ESC untuk keluar
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Player 1 activated SHIELD!
Player 1 shield deactivated.
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 activated SHIELD!
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactivated.
Player 1 shield deactiv

TypeError: 'NoneType' object is not subscriptable

: 